# 02 - Data Cleaning and Quality Validation

## Overview
This notebook covers:
- Handling missing values
- Removing duplicates
- Detecting and handling outliers
- Data validation (ranges, types)
- Quality metrics

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import json

print("Imports successful")

Imports successful


In [2]:
# Setup paths
PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Load raw data
raw_file = DATA_RAW / 'raw_smart_meter.csv'
df = pd.read_csv(raw_file)
df_clean = df.copy()

print(f"Loaded {len(df):,} records")

Loaded 25,000 records


## 1. Record Original Metrics

In [3]:
# Record original state
original_count = len(df_clean)
original_nulls = df_clean.isnull().sum().sum()
original_duplicates = df_clean.duplicated().sum()

print("\n=== ORIGINAL DATA STATE ===")
print(f"Total records: {original_count:,}")
print(f"Total null values: {original_nulls:,}")
print(f"Duplicate records: {original_duplicates}")


=== ORIGINAL DATA STATE ===
Total records: 25,000
Total null values: 500
Duplicate records: 600


## 2. Handle Missing Values

In [4]:
# Identify null values by column
null_cols = df_clean.isnull().sum()
null_cols = null_cols[null_cols > 0].sort_values(ascending=False)

if len(null_cols) > 0:
    print("\n=== NULL VALUES BY COLUMN ===")
    print(null_cols)
    
    # Handle numerical columns with median imputation
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df_clean[col].isnull().sum() > 0:
            median_val = df_clean[col].median()
            df_clean[col].fillna(median_val, inplace=True)
            print(f"  {col}: filled with median {median_val:.2f}")
    
    # Handle categorical columns with mode
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if df_clean[col].isnull().sum() > 0:
            mode_val = df_clean[col].mode()[0] if len(df_clean[col].mode()) > 0 else 'Unknown'
            df_clean[col].fillna(mode_val, inplace=True)
            print(f"  {col}: filled with mode {mode_val}")
else:
    print("\n[OK] No null values found")


=== NULL VALUES BY COLUMN ===
Voltage_V          250
Active_Power_kW    250
dtype: int64
  Voltage_V: filled with median 229.34
  Active_Power_kW: filled with median 1.76


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_71900\200181393.py:14: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_clean[col].fillna(median_val, inplace=True)
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_71900\200181393.py:14: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assign

## 3. Remove Duplicates

In [5]:
# Remove complete duplicates
before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates()
after_dedup = len(df_clean)
duplicates_removed = before_dedup - after_dedup

print(f"\n=== DUPLICATE REMOVAL ===")
print(f"Before: {before_dedup:,} records")
print(f"After: {after_dedup:,} records")
print(f"Removed: {duplicates_removed} duplicates")

if duplicates_removed == 0:
    print("[OK] No duplicates found")


=== DUPLICATE REMOVAL ===
Before: 25,000 records
After: 24,400 records
Removed: 600 duplicates


## 4. Detect and Handle Outliers

In [6]:
# Detect outliers using IQR method for numeric columns
print("\n=== OUTLIER DETECTION (IQR METHOD) ===")
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
outlier_flags = pd.DataFrame(index=df_clean.index)

total_outliers = 0
for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = ((df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)).sum()
    if outliers > 0:
        print(f"  {col}: {outliers} outliers (bounds: {lower_bound:.2f} to {upper_bound:.2f})")
        total_outliers += outliers
        outlier_flags[col] = (df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)

if total_outliers == 0:
    print("  No significant outliers detected")
    
# Add outlier flag column
df_clean['is_outlier'] = outlier_flags.any(axis=1) if len(outlier_flags.columns) > 0 else False
print(f"\nTotal records flagged as outliers: {df_clean['is_outlier'].sum()}")


=== OUTLIER DETECTION (IQR METHOD) ===
  Voltage_V: 222 outliers (bounds: 219.03 to 239.72)
  Current_A: 255 outliers (bounds: -7.78 to 27.83)
  Active_Power_kW: 214 outliers (bounds: -1.59 to 5.78)
  Reactive_Power_kW: 671 outliers (bounds: -0.83 to 2.58)
  Apparent_Power_kVA: 243 outliers (bounds: -1.77 to 6.34)
  Frequency_Hz: 135 outliers (bounds: 49.88 to 50.12)
  Sub_Meter_HVAC: 177 outliers (bounds: -2.52 to 4.63)

Total records flagged as outliers: 1108


## 5. Data Validation

In [7]:
# Validate data ranges and values
print("\n=== DATA VALIDATION ===")

# Timestamp validation
if 'Timestamp' in df_clean.columns:
    df_clean['Timestamp'] = pd.to_datetime(df_clean['Timestamp'], errors='coerce')
    print(f"[OK] Timestamp: {df_clean['Timestamp'].min()} to {df_clean['Timestamp'].max()}")

# Voltage validation (typical 200-250V)
if 'Voltage_V' in df_clean.columns:
    invalid_voltage = ((df_clean['Voltage_V'] < 180) | (df_clean['Voltage_V'] > 270)).sum()
    print(f"  Voltage: {invalid_voltage} records outside normal range (180-270V)")

# Frequency validation (typically 48-52 Hz)
if 'Frequency_Hz' in df_clean.columns:
    invalid_freq = ((df_clean['Frequency_Hz'] < 48) | (df_clean['Frequency_Hz'] > 52)).sum()
    print(f"  Frequency: {invalid_freq} records outside normal range (48-52Hz)")

# Power must be positive
if 'Active_Power_kW' in df_clean.columns:
    negative_power = (df_clean['Active_Power_kW'] < 0).sum()
    print(f"  Active Power: {negative_power} negative records")


=== DATA VALIDATION ===
[OK] Timestamp: 2021-01-01 00:00:00 to 2021-12-31 23:00:00
  Voltage: 100 records outside normal range (180-270V)
  Frequency: 0 records outside normal range (48-52Hz)
  Active Power: 0 negative records


## 6. Data Standardization

In [8]:
# Standardize data types
print("\n=== DATA STANDARDIZATION ===")

# Ensure numeric columns are float
numeric_cols_to_convert = ['Active_Power_kW', 'Reactive_Power_kVAR', 'Apparent_Power_kVA', 
                            'Voltage_V', 'Current_A', 'Frequency_Hz']

for col in numeric_cols_to_convert:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        print(f"  {col}: converted to float64")

# Ensure Meter_ID and Zone are strings
if 'Meter_ID' in df_clean.columns:
    df_clean['Meter_ID'] = df_clean['Meter_ID'].astype(str)
if 'Zone' in df_clean.columns:
    df_clean['Zone'] = df_clean['Zone'].astype(str)

    print("\nÃ¢Å“â€œ Data types standardized")


=== DATA STANDARDIZATION ===
  Active_Power_kW: converted to float64
  Apparent_Power_kVA: converted to float64
  Voltage_V: converted to float64
  Current_A: converted to float64
  Frequency_Hz: converted to float64


## 7. Quality Metrics Report

In [9]:
# Generate quality report
quality_metrics = {
    'timestamp': datetime.now().isoformat(),
    'original_records': original_count,
    'cleaned_records': len(df_clean),
    'records_removed': original_count - len(df_clean),
    'removal_percentage': round((original_count - len(df_clean)) / original_count * 100, 2),
    'original_nulls': int(original_nulls),
    'final_nulls': int(df_clean.isnull().sum().sum()),
    'duplicates_removed': int(duplicates_removed),
    'outliers_flagged': int(df_clean['is_outlier'].sum()),
    'data_quality_score': round((len(df_clean) / original_count) * 100, 2)
}

print("\n=== DATA QUALITY REPORT ===")
for key, value in quality_metrics.items():
    print(f"{key}: {value}")


=== DATA QUALITY REPORT ===
timestamp: 2026-04-19T04:13:40.927687
original_records: 25000
cleaned_records: 24400
records_removed: 600
removal_percentage: 2.4
original_nulls: 500
final_nulls: 500
duplicates_removed: 600
outliers_flagged: 1108
data_quality_score: 97.6


## 8. Export Cleaned Data

In [10]:
# Save cleaned data
output_file = DATA_PROCESSED / 'output_cleaned.csv'
df_clean.to_csv(output_file, index=False)

print(f"\n[OK] Cleaned data saved to: {output_file}")
print(f"[OK] Records: {len(df_clean):,}")
print(f"[OK] Columns: {len(df_clean.columns)}")

# Save quality metrics
metrics_file = DATA_PROCESSED / 'cleaning_metrics.json'
with open(metrics_file, 'w') as f:
    json.dump(quality_metrics, f, indent=2)

print(f"[OK] Quality metrics saved to: {metrics_file}")


[OK] Cleaned data saved to: C:\Smart Meter Data Systems\data\processed\output_cleaned.csv
[OK] Records: 24,400
[OK] Columns: 13
[OK] Quality metrics saved to: C:\Smart Meter Data Systems\data\processed\cleaning_metrics.json


## 9. Summary

In [11]:
print("\n" + "="*80)
print("DATA CLEANING SUMMARY")
print("="*80)
print(f"[OK] Original records: {original_count:,}")
print(f"[OK] Cleaned records: {len(df_clean):,}")
print(f"[OK] Data quality score: {quality_metrics['data_quality_score']}%")
print(f"[OK] Cleaned data exported")
print(f"\nNext: Proceed to feature engineering notebook")
print("="*80)


DATA CLEANING SUMMARY
[OK] Original records: 25,000
[OK] Cleaned records: 24,400
[OK] Data quality score: 97.6%
[OK] Cleaned data exported

Next: Proceed to feature engineering notebook
